# Task 4: Email Spam Detection with ML

**Author**: Pratham Bhat  
**Objective**: Detect spam emails using TF-IDF and Naive Bayes  
**Dataset**: Spam email data (user-provided CSV)  
**Technique**: Text vectorization (TF-IDF), Multinomial Naive Bayes classification

## 1. Load Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# COLUMN NAME CONFIG - Edit if your CSV has different column names
LABEL_COLUMN = 'label'  # or 'v1'
MESSAGE_COLUMN = 'message'  # or 'v2'

# Load data
df = pd.read_csv('spam.csv', encoding='latin-1')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3))

## 2. Data Preprocessing

In [ ]:
# Select relevant columns
df = df[[LABEL_COLUMN, MESSAGE_COLUMN]].copy()

# Convert labels
df[LABEL_COLUMN] = df[LABEL_COLUMN].map({'spam': 1, 'ham': 0})
df = df.dropna()

labels_text = ['Ham', 'Spam']
print(f"Preprocessed shape: {df.shape}")
print(f"\nClass distribution:")
print(df[LABEL_COLUMN].value_counts())
print(f"\nPercentages:")
print(df[LABEL_COLUMN].value_counts(normalize=True) * 100)

## 3. Feature Engineering & EDA

In [ ]:
# Message statistics
df['message_length'] = df[MESSAGE_COLUMN].apply(len)
df['word_count'] = df[MESSAGE_COLUMN].apply(lambda x: len(str(x).split()))

print("Message Statistics:")
print(df[['message_length', 'word_count']].describe())

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Count
counts = df[LABEL_COLUMN].value_counts().sort_index().values
colors = ['green', 'red']
axes[0].bar(labels_text, counts, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Email Distribution', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Length distribution
for label, color in [(0, 'green'), (1, 'red')]:
    axes[1].hist(df[df[LABEL_COLUMN] == label]['message_length'], 
                alpha=0.6, label=labels_text[label], color=color, bins=30)
axes[1].set_xlabel('Message Length', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('Message Length Distribution', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Word count distribution
for label, color in [(0, 'green'), (1, 'red')]:
    axes[2].hist(df[df[LABEL_COLUMN] == label]['word_count'], 
                alpha=0.6, label=labels_text[label], color=color, bins=30)
axes[2].set_xlabel('Word Count', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[2].set_title('Word Count Distribution', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. TF-IDF Vectorization & Model Training

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', lowercase=True)
X = tfidf.fit_transform(df[MESSAGE_COLUMN])
y = df[LABEL_COLUMN].values

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predictions
y_pred = nb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=labels_text))

## 5. Model Evaluation

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=labels_text, yticklabels=labels_text,
            cbar_kws={"shrink": 0.8}, linewidths=2, linecolor='black')
ax.set_title(f'Confusion Matrix (Accuracy: {accuracy:.4f})', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
y_pred_proba = nb_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
ax.set_title('ROC Curve - Spam Detection', fontsize=14, fontweight='bold')
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary

**Model Performance:**
- Accuracy indicates overall correctness
- ROC-AUC shows classification ability
- Confusion matrix shows false positives and false negatives